# Workbench

An interactive front end to the benchmarking suite, using the same pipeline as `python run.py ...`.

| you edit | you don't edit |
|---|---|
| `approaches/*.py`: cut generators and branching rules (start from `python run.py new-approach NAME`) | `src/branchcut/`: solver, baselines, instance families, validation, benchmark runner, report |
| `configs/*.toml`: experiments (instances × strategies × branching × seeds) | `tests/`: the pipeline's tests plus the approach gate |

Typical loop: edit an approach file, then run **1** until it passes, add it to a config and run **2**, and dig into the results with **3** and **4**. Re-run the setup cell after editing an approach; files are re-imported on every `discover()` call.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "run.py").exists() else Path.cwd().parent
os.chdir(ROOT)                                   # configs/, data/ and results/ are relative to the repo root
sys.path.insert(0, str(ROOT / "src"))

from IPython.display import Markdown, display

from branchcut.pipeline.benchmark import run_benchmark
from branchcut.pipeline.config import load_config
from branchcut.pipeline.registry import discover
from branchcut.pipeline.report import annotate, load_runs
from branchcut.pipeline.validation import validate
from branchcut.scenarios.generators import generate
from branchcut.solver import GurobiSolver, RunSpec

solver = GurobiSolver()
registry = discover("approaches")
print("Gurobi", solver.version)
print(registry.describe())

Gurobi 13.0.3
cut generators:
  cmir                     approaches/cmir.py
  cover                    baseline
  gmi                      baseline
branching rules:
  default                  gurobi
  max_infeasibility        gurobi
  most_fractional          gurobi
  objective_priority       approaches/objective_priority.py
  pseudo_reduced_cost      gurobi
  pseudo_shadow_price      gurobi
  strong                   gurobi
special cut strategies: backend (Gurobi's own cuts), none (no cuts)


## 1. Validate an approach

These are the same checks `python run.py validate NAME` runs and every benchmark runs first: cut validity on fully enumerated instances, separation, no leaked Gurobi cuts, a sound root bound, and the correct optimum on knapsack, set-cover and facility-location instances.

In [2]:
APPROACH = "cmir"                                # any approach name listed above
registry = discover("approaches")                # picks up edits to approaches/*.py
for report in validate([APPROACH], registry, solver):
    print(report)

== cut generator cmir: PASSED

instance                            config                 obj   nodes time(s)   cuts root gap   Gurobi cuts in log
mkp_n60_m8_s7                       backend               1609    2177    0.21      0    15.8%   {'Gomory': 1, 'Lift-and-project': 12, 'Cover': 33, 'MIR': 6, 'StrongCG': 2, 'Zero half': 2}
mkp_n60_m8_s7                       none                  1609    2200    0.13      0     0.0%   
mkp_n60_m8_s7                       cmir                  1609    1794    0.85    121     5.1%   
scp_r100_c300_d0.05_mc3_s2          backend                 23      16    0.59      0    19.1%   {'Gomory': 3, 'MIR': 6, 'Zero half': 6, 'Mod-K': 1}
scp_r100_c300_d0.05_mc3_s2          none                    23      39    0.06      0     0.0%   
scp_r100_c300_d0.05_mc3_s2          cmir                    23      39    0.12      0     0.0%   
cfl_f8_c24_s1                       backend            3023.33      12    0.05      0    85.4%   {'Gomory': 1, 'Implied bou

## 2. Run a benchmark config

Equivalent to `python run.py bench configs/smoke.toml`. Results go to `results/<name>/<timestamp>/` (`runs.jsonl`, `summary.md`, `runs.csv`, `config.toml`, `environment.json`). Below are the overview and per-family tables; the per-instance table is in `summary.md`.

In [3]:
config = load_config("configs/smoke.toml")
run_dir = run_benchmark(config, registry, solver, progress=lambda line: None)   # progress=print to follow along
summary = (run_dir / "summary.md").read_text()
display(Markdown(summary.split("## Per instance")[0]))
print(f"full summary: {run_dir / 'summary.md'}")

# smoke

Small instances from each generated family; every baseline, the example approaches, and a variant.

* results: `results/smoke/20261023-164903`
* runs: 112 on 7 instances, seeds [0], time limit 30 s
* baseline: `none`, reference: `backend`
* solver: gurobi 13.0.3, commit c45ff72ae0 (dirty)

## Overview

| strategy | runs | optimal | time limit | errors | wrong | sgm time (s) | time vs baseline | sgm nodes | nodes vs baseline | sgm work | root gap closed (%) | cuts added | sep share (%) | time vs default |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| backend / default | 7 | 7 | 0 | 0 | 0 | 0.0614 | 1.95 | 31.8 | 0.646 | 0.0527 | 49.8 | 0 | 0 | 1 |
| backend / objective_priority | 7 | 7 | 0 | 0 | 0 | 0.0741 | 2.06 | 49.7 | 0.832 | 0.065 | 49.8 | 0 | 0 | 1.21 |
| none / default | 7 | 7 | 0 | 0 | 0 | 0.0315 | 1 | 49.3 | 1 | 0.0121 | 0 | 0 | 0 | 1 |
| none / objective_priority | 7 | 7 | 0 | 0 | 0 | 0.0359 | 1 | 59.7 | 1 | 0.0151 | 0 | 0 | 0 | 1.14 |
| gmi / default | 7 | 7 | 0 | 0 | 0 | 0.24 | 7.62 | 78.7 | 1.6 | 0.182 | 36.4 | 187 | 8.29 | 1 |
| gmi / objective_priority | 7 | 7 | 0 | 0 | 0 | 0.272 | 7.59 | 92.8 | 1.55 | 0.251 | 36.4 | 187 | 8.19 | 1.14 |
| cover / default | 7 | 7 | 0 | 0 | 0 | 0.0623 | 1.98 | 47.5 | 0.963 | 0.0166 | 3.08 | 55 | 15.4 | 1 |
| cover / objective_priority | 7 | 7 | 0 | 0 | 0 | 0.0556 | 1.55 | 56.9 | 0.952 | 0.0195 | 3.08 | 60.3 | 18.8 | 0.893 |
| gmi+cover / default | 7 | 7 | 0 | 0 | 0 | 0.286 | 9.09 | 74.7 | 1.51 | 0.204 | 36.4 | 281 | 14.6 | 1 |
| gmi+cover / objective_priority | 7 | 7 | 0 | 0 | 0 | 0.315 | 8.76 | 97.9 | 1.64 | 0.276 | 36.4 | 289 | 17.6 | 1.1 |
| cmir / default | 7 | 7 | 0 | 0 | 0 | 0.059 | 1.87 | 47.3 | 0.959 | 0.0254 | 6.61 | 57.4 | 16.9 | 1 |
| cmir / objective_priority | 7 | 7 | 0 | 0 | 0 | 0.0856 | 2.38 | 65.1 | 1.09 | 0.0355 | 6.61 | 55.9 | 20.3 | 1.45 |
| gmi_3_rounds / default | 7 | 7 | 0 | 0 | 0 | 0.0733 | 2.33 | 63.2 | 1.28 | 0.0613 | 30.4 | 54.9 | 6.63 | 1 |
| gmi_3_rounds / objective_priority | 7 | 7 | 0 | 0 | 0 | 0.0747 | 2.08 | 68 | 1.14 | 0.0672 | 30.4 | 54.9 | 6.39 | 1.02 |
| backend+cmir / default | 7 | 7 | 0 | 0 | 0 | 0.129 | 4.09 | 38.5 | 0.781 | 0.0753 | 48.8 | 40.1 | 20 | 1 |
| backend+cmir / objective_priority | 7 | 7 | 0 | 0 | 0 | 0.128 | 3.57 | 49.1 | 0.821 | 0.0788 | 48.8 | 37.4 | 21.7 | 0.994 |

## By family: sgm time (s)

| strategy | facility_location | knapsack | set_cover |
|---|---|---|---|
| backend / default | 0.043 | 0.0584 | 0.0848 |
| backend / objective_priority | 0.0438 | 0.0784 | 0.0984 |
| none / default | 0.0249 | 0.0345 | 0.0336 |
| none / objective_priority | 0.0235 | 0.0412 | 0.0405 |
| gmi / default | 0.0833 | 0.0641 | 0.786 |
| gmi / objective_priority | 0.0753 | 0.0856 | 0.911 |
| cover / default | 0.0722 | 0.0567 | 0.0608 |
| cover / objective_priority | 0.0228 | 0.0671 | 0.072 |
| gmi+cover / default | 0.0843 | 0.114 | 0.891 |
| gmi+cover / objective_priority | 0.0735 | 0.124 | 1.04 |
| cmir / default | 0.0209 | 0.0937 | 0.0465 |
| cmir / objective_priority | 0.0205 | 0.146 | 0.0653 |
| gmi_3_rounds / default | 0.0296 | 0.0388 | 0.175 |
| gmi_3_rounds / objective_priority | 0.0282 | 0.0391 | 0.182 |
| backend+cmir / default | 0.0435 | 0.2 | 0.115 |
| backend+cmir / objective_priority | 0.0404 | 0.177 | 0.148 |

## By family: mean root gap closed (%)

| strategy | facility_location | knapsack | set_cover |
|---|---|---|---|
| backend / default | 86.6 | 45.4 | 19.6 |
| backend / objective_priority | 86.6 | 45.4 | 19.6 |
| none / default | 0 | 0 | 0 |
| none / objective_priority | 0 | 0 | 0 |
| gmi / default | 59.5 | 33.1 | 18.2 |
| gmi / objective_priority | 59.5 | 33.1 | 18.2 |
| cover / default | 0 | 7.2 | 0 |
| cover / objective_priority | 0 | 7.2 | 0 |
| gmi+cover / default | 59.5 | 33.1 | 18.2 |
| gmi+cover / objective_priority | 59.5 | 33.1 | 18.2 |
| cmir / default | 0 | 15.4 | 0 |
| cmir / objective_priority | 0 | 15.4 | 0 |
| gmi_3_rounds / default | 54 | 24.3 | 15.8 |
| gmi_3_rounds / objective_priority | 54 | 24.3 | 15.8 |
| backend+cmir / default | 86.6 | 43 | 19.6 |
| backend+cmir / objective_priority | 86.6 | 43 | 19.6 |



full summary: results/smoke/20261023-164903/summary.md


## 3. Explore results

Each row of `runs.jsonl` is a dict with every recorded metric. `annotate` adds `best_known`, `wrong` and `gap_closed`. To load an older run, use `load_runs("results/<name>/<timestamp>")`.

In [4]:
runs = load_runs(run_dir)
annotate(runs, reference=config.reference)


def compare(a, b, metric="nodes", branching="default"):
    # per instance and seed: metric for strategy a vs strategy b
    table = {}
    for r in runs:
        if r["branching"] == branching and r["strategy"] in (a, b) and r[metric] is not None:
            table.setdefault((r["instance"], r["seed"]), {})[r["strategy"]] = r[metric]
    print(f"{metric}: {a} vs {b} (branching={branching})")
    print(f"{'instance':<32}{'seed':>5}{a:>14}{b:>14}")
    for (inst, seed), v in sorted(table.items()):
        if a in v and b in v:
            print(f"{inst:<32}{seed:>5}{v[a]:>14.4g}{v[b]:>14.4g}")


compare("none", "cmir", "nodes")
print()
compare("gmi_3_rounds", "gmi", "gap_closed")

nodes: none vs cmir (branching=default)
instance                         seed          none          cmir
cfl_f10_c30_s0                      0            22            22
cfl_f10_c30_s1                      0            25            25
mkp_n40_m5_s100                     0           720           491
mkp_n40_m5_s101                     0           152           147
mkp_n40_m5_s102                     0            18            23
scp_r100_c300_d0.05_mc3_s0          0            31            31
scp_r100_c300_d0.05_mc3_s1          0             7             7

gap_closed: gmi_3_rounds vs gmi (branching=default)
instance                         seed  gmi_3_rounds           gmi
cfl_f10_c30_s0                      0        0.4864        0.5089
cfl_f10_c30_s1                      0        0.5932        0.6821
mkp_n40_m5_s100                     0        0.1601        0.2782
mkp_n40_m5_s101                     0        0.2187        0.2811
mkp_n40_m5_s102                     0        0.35

## 4. Debug a single run

Solve one instance with `record_cuts=True` to inspect every cut: the point it was separated from, where it was generated, and its violation.

In [5]:
inst = generate("knapsack", 7, "data/generated/knapsack", n=60, m=8)
result = solver.solve(RunSpec(inst, [registry.generators[APPROACH]()], record_cuts=True,
                              solver_params={"Threads": 1}))
print(result.status, result.objective, "nodes", result.nodes, "generated", result.cuts_generated,
      "applied by Gurobi", result.user_cuts_applied)
first = result.cut_log[0]
coef = dict(list(first["cut"].coefficients.items())[:6])
print(f"first cut ({first['where']}): {coef} ... <= {first['cut'].rhs:.4g}, "
      f"violation at its point {first['cut'].violation(first['point']):.4g}")

optimal 1609.0 nodes 1794 generated {'cmir': 7195} applied by Gurobi 121
first cut (node): {0: 3.0, 1: 1.5, 2: 2.0, 3: 2.0, 4: 2.0, 5: 3.0} ... <= 64, violation at its point 0.1817
